# regression tunner

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
tf.keras.utils.set_random_seed(0)

def plot_history(history: tf.keras.callbacks.History) -> None:
    """Plot training and validation loss/metrics curves."""
    hist = pd.DataFrame(history.history)
    hist['epoch'] = history.epoch
    
    # Check if mae is available
    has_mae = 'mae' in hist.columns
    n_plots = 2 if has_mae else 1
    
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    
    # Loss plot
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (MSE)')
    axes[0].plot(hist['epoch'], hist['loss'], label='Train')
    axes[0].plot(hist['epoch'], hist['val_loss'], label='Validation')
    axes[0].legend()
    axes[0].grid(True)
    
    # MAE plot (if available)
    if has_mae:
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('MAE')
        axes[1].plot(hist['epoch'], hist['mae'], label='Train')
        axes[1].plot(hist['epoch'], hist['val_mae'], label='Validation')
        axes[1].legend()
        axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()

I0000 00:00:1791711744.610480   98787 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791711744.647926   98787 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791711746.026912   98787 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
df_train = pd.read_csv(
    "https://storage.googleapis.com/download.tensorflow.org/data/abalone_train.csv",
    names=["Length", "Diameter", "Height", "Whole weight", "Shucked weight",
           "Viscera weight", "Shell weight", "Age"])
df_train.head()

,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Age
0,0.435,0.335,0.110,0.334,0.1355,0.0775,0.0965,7
1,0.585,0.450,0.125,0.874,0.3545,0.2075,0.2250,6
2,0.655,0.510,0.160,1.092,0.3960,0.2825,0.3700,14
3,0.545,0.425,0.125,0.768,0.2940,0.1495,0.2600,16
4,0.545,0.420,0.130,0.879,0.3740,0.1695,0.2300,13


In [3]:
df_train.describe()

,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Age
count,3320.000000,3320.000000,3320.000000,3320.000000,3320.000000,3320.000000,3320.000000,3320.000000
mean,0.522693,0.406575,0.139271,0.824734,0.357705,0.180162,0.237921,9.896988
std,0.121164,0.100120,0.042708,0.491182,0.222223,0.110182,0.140261,3.205654
min,0.075000,0.055000,0.000000,0.002000,0.001000,0.000500,0.001500,1.000000
25%,0.450000,0.345000,0.115000,0.436375,0.181500,0.092000,0.127375,8.000000
50%,0.540000,0.425000,0.140000,0.795250,0.335500,0.170750,0.230000,9.000000
75%,0.615000,0.480000,0.165000,1.150000,0.504500,0.253125,0.325000,11.000000
max,0.815000,0.650000,1.130000,2.825500,1.488000,0.760000,1.005000,27.000000


In [4]:
y_train = df_train.pop('Age')
X_train = df_train.copy()

In [5]:
df_test = pd.read_csv(
    "https://storage.googleapis.com/download.tensorflow.org/data/abalone_test.csv",
    names=["Length", "Diameter", "Height", "Whole weight", "Shucked weight",
           "Viscera weight", "Shell weight", "Age"])
y_test = df_test.pop('Age')
X_test = df_test.copy()

In [31]:
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')
print("Clases:", np.unique(y_train))
print("Número de clases:", len(np.unique(y_train)))

X_train shape: (3320, 7), X_test shape: (850, 7)
Clases: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 27]
Número de clases: 26


In [35]:
model = keras.Sequential()

model.add(layers.Dense(124, input_shape=(4,), activation='sigmoid'))
# output layer
model.add(layers.Dense(26, activation='sigmoid'))

## model summary
model.summary()

/home/jerry/anaconda3/lib/python3.14/site-packages/keras/src/layers/core/dense.py:100: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_26 (Dense)                │ (None, 124)            │           620 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ (None, 26)             │         3,250 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,870 (15.12 KB)

 Trainable params: 3,870 (15.12 KB)

 Non-trainable params: 0 (0.00 B)

In [36]:
# Using the Functional API
# 1. Define the input layer
inputs = keras.Input(shape=(7,))

# 2. Build the hidden layers, You can also add dropout or other layer types
x = layers.Dense(64, activation='sigmoid')(inputs)

x = layers.Dropout(0.5)(x)
x_2 = layers.Dense(64, activation='sigmoid')(x)

# 3. Define the output layer
outputs = layers.Dense(26, activation='sigmoid')(x_2)

# 4. Create the model by specifying inputs and outputs
model = keras.Model(inputs=inputs, outputs=outputs, name='functional_model')

# Model summary
model.summary()

Model: "functional_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 7)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 64)             │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ (None, 26)             │         1,690 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,362 (24.85 KB)

 Trainable params: 6,362 (24.85 KB)

 Non-trainable params: 0 (0.00 B)

In [37]:
# Early stopping callback 
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',  # can be 'val_accuracy'
    patience=5,  # if during 5 epochs there is no improvement in `val_loss`, the execution will stop
    restore_best_weights=True,
    verbose=1)

In [38]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    validation_split=0.2,
    batch_size=32,
    callbacks=[early_stopping]  # early stopping
    )

Epoch 1/50
 1/83 ━━━━━━━━━━━━━━━━━━━━ 43s 529ms/step - accuracy: 0.0312 - loss: 3.4697

W0000 00:00:1791713309.255897  100640 op_kernel.cc:1934] OP_REQUIRES failed at sparse_xent_op.cc:105 : INVALID_ARGUMENT: Received a label value of 27 which is outside the valid range of [0, 26).  Label values: 13 14 10 8 6 9 11 10 11 14 9 13 14 13 5 19 10 11 27 10 8 18 10 9 9 9 7 7 11 11 11 6
=== Source Location Trace: ===
tensorflow/core/kernels/sparse_xent_op.cc:45



InvalidArgumentError: Graph execution error:

Detected at node compile_loss/sparse_categorical_crossentropy/SparseSoftmaxCrossEntropyWithLogits/SparseSoftmaxCrossEntropyWithLogits defined at (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/home/jerry/anaconda3/lib/python3.14/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/jerry/anaconda3/lib/python3.14/site-packages/traitlets/config/application.py", line 1082, in launch_instance

  File "/home/jerry/anaconda3/lib/python3.14/site-packages/ipykernel/kernelapp.py", line 739, in start

  File "/home/jerry/anaconda3/lib/python3.14/site-packages/tornado/platform/asyncio.py", line 211, in start

  File "/home/jerry/anaconda3/lib/python3.14/asyncio/base_events.py", line 677, in run_forever

  File "/home/jerry/anaconda3/lib/python3.14/asyncio/base_events.py", line 2056, in _run_once

  File "/home/jerry/anaconda3/lib/python3.14